In [1]:
from variable_utils import *

:: loading settings :: url = jar:file:/home/leolp/%c3%81rea%20de%20trabalho/Portfolio/projeto_1_injestao_batch/venv_projeto_1/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/leolp/.ivy2/cache
The jars for the packages stored in: /home/leolp/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-d1a0a9d0-354f-4d96-a94c-7ea10bb3cf4d;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.3.0 in central
	found io.delta#delta-storage;3.3.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 130ms :: artifacts dl 7ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.3.0 from central in [default]
	io.delta#delta-storage;3.3.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   

In [2]:
def ingest_bronze_to_silver() -> None:
    BRONZE_PATH = f"{DATALAKE_PATH}/bronze/transaction_data"
    SILVER_PATH = f"{DATALAKE_PATH}/silver/transaction_data"

    df_bronze = DeltaTable.forPath(spark, BRONZE_PATH)

    try:
        df_silver = DeltaTable.forPath(spark, SILVER_PATH)
        df_silver.toDF().limit(1)

        MAIOR_DATA_SILVER = (
            df_silver
            .toDF()
            .agg(
                F.max('transaction_date')
            )
            .collect()[0][0]
        )

        df_bronze.delete(f"transaction_date < '{MAIOR_DATA_SILVER}'")

        (
            df_silver.alias('old_data')
            .merge(
                df_bronze.toDF().alias('new_data'),
                "old_data.transaction_id = new_data.transaction_id"
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )

    except Exception as e:
        if 'DELTA_MISSING_DELTA_TABLE' in str(e):
            (
                df_bronze
                .toDF()
                .write
                .format('delta')
                .option('mergeSchema', 'true')
                .mode('overwrite')
                .save(SILVER_PATH)
            )
        else:
            raise e

ingest_bronze_to_silver()

26/05/17 20:53:48 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [3]:
(
    DeltaTable.forPath(spark, '/home/leolp/Área de trabalho/Portfolio/datalake/silver/transaction_data')
    .toDF()
    .groupBy('transaction_date')
    .agg(
        F.count('transaction_date')
    )
    .show()
)

+----------------+-----------------------+
|transaction_date|count(transaction_date)|
+----------------+-----------------------+
|      2026-05-17|                    100|
+----------------+-----------------------+

